In [11]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

In [12]:
df = pd.read_csv(r'full_data_2.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 32 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      966 non-null    int64  
 1   Name            966 non-null    str    
 2   Price           337 non-null    float64
 3   Screen Size     865 non-null    float64
 4   Display         966 non-null    int64  
 5   Chipset         966 non-null    int64  
 6   NFC             966 non-null    int64  
 7   ROM             913 non-null    float64
 8   RAM             891 non-null    float64
 9   Battery         859 non-null    float64
 10  Refresh Rate    569 non-null    float64
 11  antutu_11       747 non-null    float64
 12  clock           747 non-null    float64
 13  gpu             747 non-null    str    
 14  perf_cores      395 non-null    float64
 15  eff_cores       389 non-null    float64
 16  perf_freq_ghz   365 non-null    float64
 17  eff_freq_ghz    365 non-null    float64
 18  O

In [13]:
def compute_correlation(df, target, method: str = "pearson", min_samples: int = 30, exclude_cols: list = None):
    """
    Tính hệ số tương quan giữa tất cả biến số với một biến mục tiêu.
 
    Tham số
    -------
    df          : DataFrame chứa dữ liệu
    target      : tên cột mục tiêu (mặc định 'antutu_11')
    method      : 'pearson', 'spearman', hoặc 'kendall'
    min_samples : số mẫu tối thiểu (sau khi bỏ NaN) để tính tương quan
    exclude_cols: danh sách cột muốn bỏ qua (ngoài target)
 
    Trả về
    ------
    DataFrame gồm: feature, correlation, p_value, n_samples, strength
    Sắp xếp theo |correlation| giảm dần.
    """
    if target not in df.columns:
        raise ValueError(f"Cột '{target}' không tồn tại trong DataFrame.")
 
    exclude = set(exclude_cols or []) | {target}
    numeric_cols = [c for c in df.select_dtypes(include="number").columns
                    if c not in exclude]
 
    target_series = df[target]
    records = []
 
    for col in numeric_cols:
        pair = df[[col, target]].dropna()
        n = len(pair)
        if n < min_samples:
            continue
 
        x, y = pair[col].values, pair[target].values
 
        if method == "pearson":
            r, p = stats.pearsonr(x, y)
        elif method == "spearman":
            r, p = stats.spearmanr(x, y)
        elif method == "kendall":
            r, p = stats.kendalltau(x, y)
        else:
            raise ValueError("method phải là 'pearson', 'spearman', hoặc 'kendall'")
 
        abs_r = abs(r)
        if   abs_r >= 0.7: strength = "rất mạnh"
        elif abs_r >= 0.5: strength = "mạnh"
        elif abs_r >= 0.3: strength = "trung bình"
        elif abs_r >= 0.1: strength = "yếu"
        else:              strength = "rất yếu"
 
        records.append({
            "feature":     col,
            "correlation": round(r, 4),
            "p_value":     round(p, 6),
            "n_samples":   n,
            "strength":    strength,
        })
 
    result = (pd.DataFrame(records)
                .assign(abs_corr=lambda d: d["correlation"].abs())
                .sort_values("abs_corr", ascending=False)
                .drop(columns="abs_corr")
                .reset_index(drop=True))
    return result

In [14]:
result = compute_correlation(df, 'antutu_11')

In [15]:
result

,feature,correlation,p_value,n_samples,strength
0,clock,0.9202,0.000000,747,rất mạnh
1,perf_freq_ghz,0.8942,0.000000,261,rất mạnh
2,Price,0.7539,0.000000,254,rất mạnh
3,rear_telephoto,0.6130,0.000000,643,mạnh
4,eff_freq_ghz,0.5782,0.000000,261,mạnh
5,RAM,0.5779,0.000000,690,mạnh
6,PPI,0.5725,0.000000,467,mạnh
7,ROM,0.5317,0.000000,700,mạnh
8,OS_Version,0.5193,0.000000,507,mạnh
9,SIM_total,0.4903,0.000000,747,trung bình
